# 12 — New Mexico Ensemble Time Series (2023–2025)

Generates annual field boundary polygons using Sentinel-2 imagery and the **ensemble** engine (Delineate-Anything + FTW + GeoAI with vote-based merge). Uses NMOSE WUCB agricultural polygon boundaries for evaluation.

This notebook runs a 3-year subset (2023–2025) suitable for interactive use. The corresponding Python script (`12_new_mexico_ensemble_timeseries.py`) runs the full 9-year range (2017–2025).

**Estimated runtime:** ~1–2 hours (3 years × 3 engines, GPU recommended).

**Prerequisites:**
```bash
pip install agribound[gee,delineate-anything,ftw,geoai]
agribound auth --project YOUR_GEE_PROJECT
```

## Configuration

In [ ]:
import json
from pathlib import Path

import geopandas as gpd

import agribound
from agribound.evaluate import evaluate

GEE_PROJECT = "YOUR_GEE_PROJECT"  # <-- Replace with your GEE project ID

NMOSE_SHAPEFILE = "../NMOSE Field Boundaries/WUCB ag polys.shp"
OUTPUT_DIR = Path("../../outputs/new_mexico_ensemble")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

YEARS = range(2023, 2026)
SOURCE = "sentinel2"
ENGINES = ["delineate-anything", "ftw", "geoai"]

## Derive Study Area and Load Reference Boundaries

In [ ]:
ref_gdf = gpd.read_file(NMOSE_SHAPEFILE)
print(f"{len(ref_gdf)} reference field polygons loaded")

bounds = ref_gdf.total_bounds
bbox_geojson = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "geometry": {
                "type": "Polygon",
                "coordinates": [
                    [
                        [bounds[0], bounds[1]],
                        [bounds[2], bounds[1]],
                        [bounds[2], bounds[3]],
                        [bounds[0], bounds[3]],
                        [bounds[0], bounds[1]],
                    ]
                ],
            },
            "properties": {"name": "NMOSE WUCB Study Area"},
        }
    ],
}
study_area = str(OUTPUT_DIR / "nm_study_area.geojson")
with open(study_area, "w") as f:
    json.dump(bbox_geojson, f)

print(f"Study area: {study_area}")
print(f"Bounds: {bounds}")

## Phase 1: Per-Engine Delineation

Run each engine individually for each year so we can compare before ensembling.

In [ ]:
per_engine_results = {}  # {year: {engine_name: gdf}}

for year in YEARS:
    print(f"\n--- Year {year} ---")
    per_engine_results[year] = {}

    for engine_name in ENGINES:
        output_path = OUTPUT_DIR / f"fields_{engine_name}_{year}.gpkg"

        if output_path.exists():
            print(f"  {engine_name}: already exists, skipping.")
            per_engine_results[year][engine_name] = gpd.read_file(output_path)
            continue

        try:
            gdf = agribound.delineate(
                study_area=study_area,
                source=SOURCE,
                year=year,
                engine=engine_name,
                output_path=str(output_path),
                gee_project=GEE_PROJECT,
                composite_method="median",
                cloud_cover_max=20,
                min_area=2500,
                simplify=2.0,
                device="auto",
            )
            per_engine_results[year][engine_name] = gdf
            print(f"  {engine_name}: {len(gdf)} fields")
        except Exception as exc:
            print(f"  {engine_name}: FAILED — {exc}")

## Phase 2: Ensemble Delineation (Vote Strategy)

Merge results from all three engines using majority-vote overlap.

In [ ]:
ensemble_results = {}

for year in YEARS:
    print(f"Ensemble for {year}...", end=" ")
    output_path = OUTPUT_DIR / f"fields_ensemble_{year}.gpkg"

    if output_path.exists():
        print("already exists, skipping.")
        ensemble_results[year] = gpd.read_file(output_path)
        continue

    try:
        gdf = agribound.delineate(
            study_area=study_area,
            source=SOURCE,
            year=year,
            engine="ensemble",
            output_path=str(output_path),
            gee_project=GEE_PROJECT,
            composite_method="median",
            cloud_cover_max=20,
            min_area=2500,
            simplify=2.0,
            device="auto",
            engine_params={
                "engines": ENGINES,
                "merge_strategy": "vote",
            },
        )
        ensemble_results[year] = gdf
        print(f"{len(gdf)} fields")
    except Exception as exc:
        print(f"FAILED — {exc}")

## Phase 3: Evaluation Against NMOSE Reference

In [ ]:
print(f"{'Year':<6} {'Engine':<25} {'Fields':>6} {'F1':>6} {'IoU':>6} {'P':>6} {'R':>6}")
print(f"{'-'*6} {'-'*25} {'-'*6} {'-'*6} {'-'*6} {'-'*6} {'-'*6}")

for year in YEARS:
    for engine_name in ENGINES:
        gdf = per_engine_results.get(year, {}).get(engine_name)
        if gdf is not None:
            m = evaluate(gdf, ref_gdf)
            print(f"{year:<6} {engine_name:<25} {len(gdf):>6} "
                  f"{m['f1']:.3f} {m['iou_mean']:.3f} "
                  f"{m['precision']:.3f} {m['recall']:.3f}")

    gdf = ensemble_results.get(year)
    if gdf is not None:
        m = evaluate(gdf, ref_gdf)
        print(f"{year:<6} {'** ensemble **':<25} {len(gdf):>6} "
              f"{m['f1']:.3f} {m['iou_mean']:.3f} "
              f"{m['precision']:.3f} {m['recall']:.3f}")

## Visualization: Ensemble vs NMOSE Reference

In [ ]:
from agribound.visualize import show_comparison

if ensemble_results:
    latest_year = max(ensemble_results.keys())
    m = show_comparison(
        [ensemble_results[latest_year], ref_gdf],
        labels=[f"Ensemble ({latest_year})", "NMOSE Reference"],
        basemap="Esri.WorldImagery",
        output_html=str(OUTPUT_DIR / "map_ensemble_vs_reference.html"),
    )
    m

## Visualization: Per-Engine Comparison

In [ ]:
if ensemble_results and per_engine_results.get(latest_year):
    engine_gdfs = list(per_engine_results[latest_year].values())
    engine_labels = list(per_engine_results[latest_year].keys())
    engine_gdfs.append(ensemble_results[latest_year])
    engine_labels.append("ensemble")

    m_engines = show_comparison(
        engine_gdfs,
        labels=engine_labels,
        basemap="Esri.WorldImagery",
        output_html=str(OUTPUT_DIR / "map_engine_comparison.html"),
    )
    m_engines

## Visualization: Ensemble Time Series

In [ ]:
ts_gdfs = [ensemble_results[y] for y in sorted(ensemble_results.keys())]
ts_labels = [str(y) for y in sorted(ensemble_results.keys())]

if len(ts_gdfs) >= 2:
    m_ts = show_comparison(
        ts_gdfs,
        labels=ts_labels,
        basemap="Esri.WorldImagery",
        output_html=str(OUTPUT_DIR / "map_ensemble_timeseries.html"),
    )
    m_ts